# Data encryption with AES

In this notebook, we show how to encrypt data using the AES cipher.

Let's start by importing the necessary stuff.

In [35]:
from Crypto.Cipher import AES

## Encrypting a simple message

To encrypt a message, we need a key.

Keys can be 128, 192, or 256 bits long (i.e., 16, 24, or 32 bytes).

In [39]:
key = b'Sixteen byte key'
message = b'The quick brown fox jumps over the lazy dog'

In addition to the message itself and the key, to encrypt the message we also need to obtain a **nonce**, i.e., a unique, non-repeating value required for every encryption operation using the same key. The nonce ensures that identical plaintexts produce different ciphertexts.

The encryption operation produces:

- A **ciphertext**.
- A **tag** ensuring the integrity and authenticity of the ciphertext.

In [40]:
cipher = AES.new(key, AES.MODE_EAX)
nonce = cipher.nonce
ciphertext, tag = cipher.encrypt_and_digest(message)
print("Ciphertext:", ciphertext.hex())
print("Tag:", tag.hex())
print("Nonce:", nonce.hex())

Ciphertext: a3b8e03df0a2ab0efbff014f14b0adb14416d472e12d7835561b5c6adf35400c2edec3f9a5fc68560c407a
Tag: bd85f132eb88c3088ce01a891090e4e9
Nonce: 0e4b3cea96aa2ae7531d96a4d8c2399d


Notice that a different key length automatically raises an error.

In [38]:
try:
  key = b'A different key'
  cipher = AES.new(key, AES.MODE_EAX)
except Exception as e:
  print(e)

Incorrect AES key length (15 bytes)


## Decrypting the message

To decrypt the message, we need:

1. The ciphertext.
2. The nonce.

...And of course the original key!

The tag can be used later for verifying the authenticity of the original message.

In [42]:
key = b'Sixteen byte key'

# Decryption
cipher = AES.new(key, AES.MODE_EAX, nonce=nonce)
plaintext = cipher.decrypt(ciphertext)
print("Plaintext:", plaintext)

# Verification
try:
  cipher.verify(tag)
  print("The message is authentic:", plaintext)
except ValueError:
  print("Key incorrect or message corrupted")

Plaintext: b'The quick brown fox jumps over the lazy dog'
The message is authentic: b'The quick brown fox jumps over the lazy dog'


## Encrypting a file

What if we want to encrypt a file?

Let us define a function to load the file into memory.

In [43]:
def load_file(file_path):
    return open(file_path, "rb").read()

Let's try an example by encrypting Shakespeare's Othello.

We need to:

1) Load the file.
2) Define a key and nonce.
3) Encrypt the original file and save the result to another file.

In [48]:
# Load the file and print its size.
file_path = "data/othello_TXT_FolgerShakespeare.txt"
message = load_file(file_path) 
print("File size:", len(message), "bytes")

# Let's reuse our previous key.
key = b'Sixteen byte key'

# Define AES instance and nonce.
cipher = AES.new(key, AES.MODE_EAX)
nonce = cipher.nonce

# Encrypt.
ciphertext, tag = cipher.encrypt_and_digest(message)

# Save ciphertext, tag and nonce to files.
ciphertext_file_path = "results/othello_encrypted_text"
tag_file_path = "results/othello_encrypted_tag"
nonce_file_path = "results/othello_encrypted_nonce"
with open(ciphertext_file_path, "wb") as file:
    file.write(ciphertext)
with open(tag_file_path, "wb") as file:
    file.write(tag)
with open(nonce_file_path, "wb") as file:
    file.write(nonce)

print("Ciphertext size:", len(ciphertext), "bytes")
print("Tag:", tag.hex())
print("Nonce:", nonce.hex())

File size: 159411 bytes
Ciphertext size: 159411 bytes
Tag: 44c9e95ffc212a6e533e6e979130b50f
Nonce: 40beebafc9b0046b8cb844aec6032b04


## Decrypting a file

To decrypt a file, we follow a dual process.

1. We load the files containing the ciphertext, tag, and nonce.
2. We define an instance of AES with the key and nonce.
3. We decrypt the ciphertext and recover the plaintext.
4. We can use the tag for verification.

In [54]:
# Load the files.
ciphertext_file_path = "results/othello_encrypted_text"
tag_file_path = "results/othello_encrypted_tag"
nonce_file_path = "results/othello_encrypted_nonce"
ciphertext = load_file(ciphertext_file_path)
tag = load_file(tag_file_path)
nonce = load_file(nonce_file_path)

# The key is always the same.
key = b'Sixteen byte key'

# Define AES instance for decryption.
cipher = AES.new(key, AES.MODE_EAX, nonce=nonce)
plaintext = cipher.decrypt(ciphertext)
print("Plaintext:", plaintext[:100], "...") # We print only the first 100 characters.

# We can use the tag for verification.
try:
    cipher.verify(tag)
    print("The message is authentic.")
except ValueError:
    print("Key incorrect or message corrupted")

Plaintext: b'Othello\r\nby William Shakespeare\r\nEdited by Barbara A. Mowat and Paul Werstine\r\n  with Michael Poston' ...
The message is authentic.
